In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.3


In [5]:
import numpy as np
import pandas as pd


raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 8],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})

print(raw)


   id name   age    city  spend        date
0   1  Ana  30.0   Pune   120.5  2024-01-05
1   2   Bo  25.0    pune   80.0  2024-01-06
2   3   Cy   NaN   DELHI  200.2  2024-01-07
3   4   Di  41.0  Delhi     N/A  2024-01-08
4   5  Eve  -1.0  Mumbai  150.0  2024-01-09
5   6  Fin  38.0  bombay  99000  2024-01-10
6   7  Gus  29.0   Pune.  110.0  2024-01-11
7   8  Gus  29.0   Pune.  110.0  2024-01-11


In [6]:
df = raw.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [7]:
print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")


Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 0

Note: spend is type object -> stored as text!


In [8]:

duplicate_count = raw.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

missing_per_col = raw.isna().sum()
print("\nMissing per column:\n", missing_per_col)



Duplicate rows: 0

Missing per column:
 id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [9]:
df = raw.copy()
df['city'] = df['city'].str.strip().str.lower()
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')
df['age']   = df['age'].replace(-1, np.nan)

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())


Missing after unmasking:
age      2
spend    1
dtype: int64


In [10]:
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())


Missing after imputing: 0


In [11]:
ex = raw.copy()

ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['age'] = ex['age'].replace(-1, np.nan)
df_dropped = ex.dropna()
df_imputed = ex.copy()
df_imputed['age'] = df_imputed['age'].fillna(df_imputed['age'].median())
df_imputed['spend'] = df_imputed['spend'].fillna(df_imputed['spend'].median())
print(f"Original row count: {len(ex)}")
print(f"Dropped rows count: {len(df_dropped)}")
print(f"Imputed rows count: {len(df_imputed)}")


Original row count: 8
Dropped rows count: 5
Imputed rows count: 8


In [12]:
print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

Before: (8, 6)
After : (8, 6) -> removed the repeated Gus row


In [13]:
df['date'] = pd.to_datetime(df['date'])
df['city'] = df['city'].astype('string')
print(df.dtypes)


id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [14]:
ex = raw.copy()
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['date'] = pd.to_datetime(ex['date'])
ex = ex.drop_duplicates()
print("Data Types:")
print(ex.dtypes)
print("\nShape:")
print(ex.shape)


Data Types:
id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object

Shape:
(8, 6)


In [15]:
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=110.0  Q3=162.6  IQR=52.6
Normal range: 31.2 to 241.4

Outlier rows:
  name    spend
5  Fin  99000.0


In [16]:
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])


  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       241.375
6  Gus    110.0       110.000
7  Gus    110.0       110.000


In [17]:

q1 = ex['age'].quantile(0.25)
q3 = ex['age'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = ex[(ex['age'] < lower_bound) | (ex['age'] > upper_bound)]
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("\nOutlier rows:\n", outliers)


Lower Bound: 16.5
Upper Bound: 44.5

Outlier rows:
    id name  age    city  spend       date
4   5  Eve -1.0  Mumbai  150.0 2024-01-09


In [18]:
print(df['city'].value_counts())

city
pune      2
delhi     2
pune.     2
mumbai    1
bombay    1
Name: count, dtype: Int64


In [19]:
s = df['city'].astype('string')
s = s.str.strip()
s = s.str.lower()
s = s.str.replace('.', '', regex=False)
s = s.replace({'bombay': 'mumbai'})
df['city'] = s
print(df['city'].value_counts())


city
pune      4
delhi     2
mumbai    2
Name: count, dtype: Int64


In [20]:
import pandas as pd

messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')
cleaned = messy.str.strip().str.lower()
cleaned = cleaned.replace({'n.y.': 'new york'})
print(cleaned.value_counts())


london      3
new york    3
Name: count, dtype: Int64


In [21]:

clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean

Final shape: (8, 6)
Missing values: 0
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,120.5,2024-01-08
4,5,Eve,29.5,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11
7,8,Gus,29.0,pune,110.0,2024-01-11
